# 05 — External Validation with Measured GFR

This notebook demonstrates the **external validation pipeline** for our eGFR
estimation models.  Because no freely downloadable dataset with gold-standard
measured GFR (iothalamate/iohexol clearance) is publicly available (see
`notebooks/external_data_sources.md` for details), we use two complementary
strategies:

1. **Synthetic mGFR dataset** — generated via `generate_synthetic_mgfr_dataset()`
   to exercise the full validation code path (P30, P10, RMSE, MAE, bias, CKD
   stage concordance).  *These results are for demonstration only and must not
   be cited as clinical evidence.*
2. **NHANES cross-equation concordance** — uses CKD-EPI 2021 as the reference
   standard and assesses agreement of MDRD, Cockcroft-Gault, and the hybrid ML
   model against it.

> **⚠️ Important:** For true external validation, submit a Data Use Agreement
> to the NIDDK Central Repository for the CRIC Study dataset, which contains
> iothalamate-measured GFR alongside serum creatinine and cystatin C.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from eGFR.data import generate_synthetic_mgfr_dataset, read_xpt, clean_kidney_data
from eGFR.models import calc_egfr_ckd_epi_2021, calc_egfr_mdrd, calc_crcl_cockcroft_gault
from eGFR.evaluate import evaluate_model, bland_altman_stats, p30_accuracy, p10_accuracy
from eGFR.utils import egfr_to_ckd_stage

warnings.simplefilter("ignore", UserWarning)
print("Imports OK")

## 1. Generate Synthetic Measured-GFR Dataset

The synthetic dataset mimics a CKD-cohort validation study with 500 patients.
Demographics are based on CRIC Study distributions; creatinine is derived from
the simulated mGFR via an inverted CKD-EPI 2021 equation with added biological
noise.

In [ ]:
df_ext = generate_synthetic_mgfr_dataset(n_samples=500, random_state=42)
print(f"Synthetic dataset: {len(df_ext)} patients")
print(f"\nColumns: {list(df_ext.columns)}")
print(f"\nmGFR range: {df_ext['measured_gfr'].min():.1f} – {df_ext['measured_gfr'].max():.1f} mL/min/1.73m²")
print(f"Creatinine range: {df_ext['cr_mgdl'].min():.2f} – {df_ext['cr_mgdl'].max():.2f} mg/dL")
print(f"\nSex distribution: males={sum(df_ext['sex']==1)}, females={sum(df_ext['sex']==2)}")
print(f"Age range: {df_ext['age_years'].min()} – {df_ext['age_years'].max()} years")
df_ext.describe().round(2)

## 2. Run All Mechanistic Equations on External Dataset

Compute eGFR/CrCl for each patient using all three equations.

In [ ]:
def safe_calc(func, *args):
    """Call func, returning NaN on error."""
    try:
        return func(*args)
    except (ValueError, TypeError):
        return float("nan")

# CKD-EPI 2021
df_ext["egfr_ckd_epi"] = df_ext.apply(
    lambda r: safe_calc(calc_egfr_ckd_epi_2021, r["cr_mgdl"], r["age_years"], r["sex"]),
    axis=1)

# MDRD
df_ext["egfr_mdrd"] = df_ext.apply(
    lambda r: safe_calc(calc_egfr_mdrd, r["cr_mgdl"], r["age_years"], r["sex"]),
    axis=1)

# Cockcroft-Gault
df_ext["crcl_cg"] = df_ext.apply(
    lambda r: safe_calc(calc_crcl_cockcroft_gault, r["cr_mgdl"], r["age_years"],
                        r["weight_kg"], r["sex"]),
    axis=1)

# Drop rows with NaN predictions
df_valid = df_ext.dropna(subset=["egfr_ckd_epi", "egfr_mdrd", "crcl_cg"]).copy()
print(f"Valid samples for evaluation: {len(df_valid)} / {len(df_ext)}")

## 3. Evaluate Each Equation Against Measured GFR

Report P30, P10, RMSE, MAE, bias, and CKD stage concordance.

In [ ]:
mgfr = df_valid["measured_gfr"].values

results = []
for name, pred_col in [("CKD-EPI 2021", "egfr_ckd_epi"),
                        ("MDRD", "egfr_mdrd"),
                        ("Cockcroft-Gault", "crcl_cg")]:
    y_pred = df_valid[pred_col].values
    metrics = evaluate_model(mgfr, y_pred, model_name=name)
    results.append(metrics)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  RMSE:    {metrics['rmse']:.2f} mL/min/1.73m²")
    print(f"  MAE:     {metrics['mae']:.2f}")
    print(f"  Bias:    {metrics['bias']:+.2f}")
    print(f"  Pearson r: {metrics['r_pearson']:.4f}")
    print(f"  P30:     {metrics['p30']:.1f}%")
    print(f"  P10:     {metrics['p10']:.1f}%")
    print(f"  CKD stage agreement: {metrics['ckd_stage_agreement']:.1f}%")

# Summary table
summary_df = pd.DataFrame([{
    "Equation": r["model_name"],
    "RMSE": round(r["rmse"], 2),
    "MAE": round(r["mae"], 2),
    "Bias": round(r["bias"], 2),
    "Pearson r": round(r["r_pearson"], 4),
    "P30 (%)": round(r["p30"], 1),
    "P10 (%)": round(r["p10"], 1),
    "CKD Agreement (%)": round(r["ckd_stage_agreement"], 1),
} for r in results])

print("\n\n" + "="*70)
print("  SUMMARY TABLE")
print("="*70)
print(summary_df.to_string(index=False))

## 4. Scatter Plots: Estimated vs Measured GFR

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, pred_col) in zip(axes, [("CKD-EPI 2021", "egfr_ckd_epi"),
                                        ("MDRD", "egfr_mdrd"),
                                        ("Cockcroft-Gault", "crcl_cg")]):
    y_pred = df_valid[pred_col].values
    ax.scatter(mgfr, y_pred, alpha=0.3, s=10, color="steelblue")
    lims = [0, max(mgfr.max(), y_pred.max()) * 1.05]
    ax.plot(lims, lims, "--", color="red", linewidth=1, label="Identity")
    ax.set_xlabel("Measured GFR (mL/min/1.73m²)")
    ax.set_ylabel(f"{name} eGFR")
    ax.set_title(name)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.legend(loc="upper left", fontsize=8)

plt.suptitle("Estimated vs Measured GFR (Synthetic Validation)", fontsize=13, y=1.02)
plt.tight_layout()
os.makedirs("../reports", exist_ok=True)
plt.savefig("../reports/external_validation_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Scatter plots saved to reports/external_validation_scatter.png")

## 5. Bland-Altman Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, pred_col) in zip(axes, [("CKD-EPI 2021", "egfr_ckd_epi"),
                                        ("MDRD", "egfr_mdrd"),
                                        ("Cockcroft-Gault", "crcl_cg")]):
    y_pred = df_valid[pred_col].values
    ba = bland_altman_stats(mgfr, y_pred)
    means = (mgfr + y_pred) / 2.0
    diffs = y_pred - mgfr

    ax.scatter(means, diffs, alpha=0.3, s=10, color="steelblue")
    ax.axhline(ba["mean_bias"], color="red", linestyle="-", linewidth=1,
               label=f"Bias={ba['mean_bias']:.1f}")
    ax.axhline(ba["loa_upper"], color="gray", linestyle="--", linewidth=0.8,
               label=f"+1.96 SD={ba['loa_upper']:.1f}")
    ax.axhline(ba["loa_lower"], color="gray", linestyle="--", linewidth=0.8,
               label=f"-1.96 SD={ba['loa_lower']:.1f}")
    ax.set_xlabel("Mean of Measured & Estimated")
    ax.set_ylabel("Estimated − Measured")
    ax.set_title(f"Bland-Altman: {name}")
    ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.savefig("../reports/external_validation_bland_altman.png", dpi=150, bbox_inches="tight")
plt.show()
print("Bland-Altman plots saved to reports/external_validation_bland_altman.png")

## 6. CKD Stage Reclassification Analysis

Compare CKD stages derived from measured GFR vs each equation:

In [ ]:
stages_mgfr = [egfr_to_ckd_stage(v) for v in mgfr]
stage_order = ["G1", "G2", "G3a", "G3b", "G4", "G5"]

for name, pred_col in [("CKD-EPI 2021", "egfr_ckd_epi"),
                        ("MDRD", "egfr_mdrd"),
                        ("Cockcroft-Gault", "crcl_cg")]:
    y_pred = df_valid[pred_col].values
    stages_pred = [egfr_to_ckd_stage(v) for v in y_pred]

    ct = pd.crosstab(
        pd.Categorical(stages_mgfr, categories=stage_order),
        pd.Categorical(stages_pred, categories=stage_order),
        rownames=["Measured GFR Stage"],
        colnames=[f"{name} Stage"],
    )
    concordant = sum(1 for s1, s2 in zip(stages_mgfr, stages_pred) if s1 == s2)
    agreement = 100.0 * concordant / len(stages_mgfr)

    print(f"\n{'='*50}")
    print(f"  {name} — CKD Stage Concordance: {agreement:.1f}%")
    print(f"{'='*50}")
    print(ct)
    print()

## 7. NHANES Cross-Equation Concordance (Fallback Validation)

When no external mGFR dataset is available, we assess internal consistency
by comparing equations against CKD-EPI 2021 (current KDIGO standard) on
NHANES data.

In [ ]:
# Try to load real NHANES data; fallback to larger synthetic dataset
nhanes_loaded = False
try:
    biopro = read_xpt("../data/raw/BIOPRO_J.XPT")
    demo = read_xpt("../data/raw/DEMO_J.XPT")
    bmx = read_xpt("../data/raw/BMX_J.XPT")
    df_nhanes = clean_kidney_data(biopro, demo, bmx)
    nhanes_loaded = True
    print(f"Loaded real NHANES 2017-2018 data: {len(df_nhanes)} samples")
except Exception as e:
    print(f"Could not load NHANES data ({e}).")
    print("Using larger synthetic dataset for concordance analysis...")
    df_nhanes = generate_synthetic_mgfr_dataset(n_samples=2000, random_state=99)

# Compute eGFR predictions
df_nhanes["egfr_ckd_epi"] = df_nhanes.apply(
    lambda r: safe_calc(calc_egfr_ckd_epi_2021, r["cr_mgdl"], r["age_years"], r["sex"]),
    axis=1)
df_nhanes["egfr_mdrd"] = df_nhanes.apply(
    lambda r: safe_calc(calc_egfr_mdrd, r["cr_mgdl"], r["age_years"], r["sex"]),
    axis=1)
if "weight_kg" in df_nhanes.columns:
    df_nhanes["crcl_cg"] = df_nhanes.apply(
        lambda r: safe_calc(calc_crcl_cockcroft_gault, r["cr_mgdl"], r["age_years"],
                            r["weight_kg"], r["sex"]),
        axis=1)

df_nh = df_nhanes.dropna(subset=["egfr_ckd_epi", "egfr_mdrd"]).copy()
ref = df_nh["egfr_ckd_epi"].values

print(f"\nCross-equation concordance (reference: CKD-EPI 2021, n={len(df_nh)})")
for name, col in [("MDRD", "egfr_mdrd"), ("Cockcroft-Gault", "crcl_cg")]:
    if col not in df_nh.columns:
        continue
    pred = df_nh[col].values
    valid = ~np.isnan(pred)
    if valid.sum() == 0:
        continue
    metrics = evaluate_model(ref[valid], pred[valid], model_name=name)
    print(f"\n  {name} vs CKD-EPI 2021:")
    print(f"    RMSE={metrics['rmse']:.2f}, MAE={metrics['mae']:.2f}, "
          f"Bias={metrics['bias']:+.2f}, P30={metrics['p30']:.1f}%, "
          f"CKD Stage Agreement={metrics['ckd_stage_agreement']:.1f}%")

## 8. Dataset Limitations

### Synthetic mGFR Dataset Limitations

- **Not real patient data** — Creatinine values were generated by inverting the
  CKD-EPI 2021 equation, introducing circular dependency.
- **Simplified noise model** — Real biological variability is more complex than
  the ±10% Gaussian noise applied here.
- **No comorbidity effects** — Diabetes, heart failure, and other conditions
  that affect the creatinine–GFR relationship are not modeled.
- **No assay variability** — Different creatinine assays have different biases.

### NHANES Cross-Equation Concordance Limitations

- **No ground truth** — NHANES does not include measured GFR (iothalamate or
  iohexol clearance), so P30/P10 relative to true GFR cannot be assessed.
- **Circular validation risk** — The hybrid ML model is partially trained on
  CKD-EPI 2021 outputs, so agreement may be inflated.
- **CrCl ≠ eGFR** — Cockcroft-Gault returns CrCl (mL/min), not eGFR
  (mL/min/1.73m²), so direct comparison is methodologically imperfect.

### Recommended Next Steps

1. Submit NIDDK DUA for **CRIC Study** data (iothalamate-measured GFR, n≈3,900)
2. Replace `generate_synthetic_mgfr_dataset()` call with real CRIC data loader
3. Re-run this notebook with real measured GFR for publication-grade validation